In [ ]:
import os
import sys

WORKDIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else str((__import__('pathlib').Path.cwd()).resolve())
REPO_URL = 'https://github.com/PhuongThao-2005/TULIP-MedML.git'
REPO_NAME = 'TULIP-MedML'
BRANCH = 'main' 

repo_path = os.path.join(WORKDIR, REPO_NAME)

if os.path.isdir(repo_path):
    os.system(f'cd {repo_path} && git fetch origin')
    os.system(f'cd {repo_path} && git checkout {BRANCH}')
    os.system(f'cd {repo_path} && git pull origin {BRANCH}')
    print(f'Pulled latest: {repo_path} [{BRANCH}]')
else:
    os.system(f'cd {WORKDIR} && git clone -b {BRANCH} {REPO_URL}')
    print(f'Cloned: {repo_path} [{BRANCH}]')

os.environ['TULIP_PROJECT_ROOT'] = repo_path
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

os.chdir(repo_path)
print('Working dir:', os.getcwd())

In [ ]:
from pathlib import Path
import os
import re
import sys
import glob
import subprocess
import yaml
import numpy as np
import pandas as pd
import torch

if os.environ.get('TULIP_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['TULIP_PROJECT_ROOT']).resolve()
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()


DATA_ROOT    = "/kaggle/input/datasets/ashery/chexpert"
TEST_CSV = "/kaggle/working/TULIP-MedML/data/test.csv"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_ROOT    =', DATA_ROOT)
print('TEST_CSV     =', TEST_CSV)
print('CUDA         =', torch.cuda.is_available())

In [ ]:
from src.data.chexpert import CheXpert, NUM_CLASSES

def load_cfg(config_path):
    with open(config_path, encoding='utf-8') as f:
        return yaml.safe_load(f)

def strip_module_prefix(state_dict):
    return {k[7:] if k.startswith('module.') else k: v for k, v in state_dict.items()}

def add_module_prefix(state_dict):
    return {k if k.startswith('module.') else f'module.{k}': v for k, v in state_dict.items()}

def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ('state_dict', 'model_state_dict', 'model'):
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]
        return checkpoint
    raise ValueError('Unsupported checkpoint format.')

def load_state_dict_flexible(model, state_dict):
    for candidate in (state_dict, strip_module_prefix(state_dict), add_module_prefix(state_dict)):
        try:
            model.load_state_dict(candidate)
            return
        except RuntimeError:
            pass
    raise RuntimeError('Cannot load checkpoint into model.')

def remap_path(path_value, project_root, data_root=None):
    if not path_value:
        return path_value

    path_str = str(path_value)
    p = Path(path_str)
    if p.exists():
        return str(p)

    kaggle_repo_prefix = '/kaggle/working/TULIP-MedML/'
    kaggle_data_prefix = '/kaggle/input/datasets/ashery/chexpert'

    if path_str.startswith(kaggle_repo_prefix):
        rel = path_str[len(kaggle_repo_prefix):]
        return str(project_root / rel)

    if data_root is not None and path_str.startswith(kaggle_data_prefix):
        rel = path_str[len(kaggle_data_prefix):].lstrip('/')
        return str(Path(data_root) / rel)

    if path_str.startswith('/') and data_root is not None and 'CheXpert' in path_str:
        return str(Path(data_root))

    if not os.path.isabs(path_str):
        return str(project_root / path_str)

    return path_str

def resolve_cfg(cfg, project_root, data_root=None, test_csv_override=None):
    cfg = dict(cfg)
    cfg['data'] = dict(cfg['data'])

    cfg['data']['root'] = remap_path(cfg['data']['root'], project_root, data_root)
    for key in ('train_csv', 'val_csv', 'val_uncertain_csv', 'word_vec', 'adj'):
        if key in cfg['data']:
            cfg['data'][key] = remap_path(cfg['data'][key], project_root, data_root)

    if test_csv_override is not None:
        cfg['data']['test_csv'] = remap_path(test_csv_override, project_root, data_root)
    elif 'test_csv' in cfg['data']:
        cfg['data']['test_csv'] = remap_path(cfg['data']['test_csv'], project_root, data_root)

    return cfg

def find_checkpoint(candidates):
    for p in candidates:
        if not p:
            continue
        if os.path.isfile(p):
            return p

    all_candidates = []
    for p in candidates:
        if p and os.path.isdir(p):
            all_candidates.extend(glob.glob(os.path.join(p, '**', 'model_best.pth.tar'), recursive=True))
            all_candidates.extend(glob.glob(os.path.join(p, '**', 'checkpoint_epoch_*.pth.tar'), recursive=True))

    if not all_candidates:
        return None

    all_candidates = sorted(set(all_candidates), key=os.path.getmtime, reverse=True)
    return all_candidates[0]

def _extract_drive_file_id(url):
    if not isinstance(url, str):
        return None

    m = re.search(r'/file/d/([a-zA-Z0-9_-]+)', url)
    if m:
        return m.group(1)

    m = re.search(r'[?&]id=([a-zA-Z0-9_-]+)', url)
    if m:
        return m.group(1)

    return None

def _ensure_gdown():
    import importlib

    try:
        return importlib.import_module('gdown')
    except ImportError:
        print('[INFO] Installing gdown for Google Drive download...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
        return importlib.import_module('gdown')

def download_checkpoint_from_drive(url, model_name='model', target_dir=None):
    file_id = _extract_drive_file_id(url)
    if file_id is None:
        raise ValueError(f'Unsupported URL format: {url}')

    if target_dir is None:
        if os.environ.get('KAGGLE_URL_BASE') or str(PROJECT_ROOT).startswith('/kaggle'):
            target_dir = Path('/kaggle/working/drive_ckpts')
        else:
            target_dir = PROJECT_ROOT / 'drive_ckpts'
    else:
        target_dir = Path(target_dir)

    target_dir.mkdir(parents=True, exist_ok=True)
    out_path = target_dir / f'{model_name}.pth.tar'

    if out_path.exists():
        print(f'[INFO] Reuse downloaded checkpoint: {out_path}')
        return str(out_path)

    gdown = _ensure_gdown()
    print(f'[INFO] Downloading Google Drive checkpoint for {model_name} ...')
    gdown.download(id=file_id, output=str(out_path), quiet=False)

    if not out_path.exists():
        raise RuntimeError(f'Download failed for {model_name} from {url}')

    return str(out_path)

def resolve_checkpoint_entry(entry, model_name='model', project_root=None):
    if entry is None:
        return None

    v = str(entry).strip()
    if not v:
        return None

    if v.startswith('http://') or v.startswith('https://'):
        if 'drive.google.com' in v:
            return download_checkpoint_from_drive(v, model_name=model_name)
        raise ValueError(f'Unsupported URL format: {v}')

    p = Path(v)
    if p.exists():
        return str(p)

    if project_root is not None and not p.is_absolute():
        p2 = Path(project_root) / p
        if p2.exists():
            return str(p2)

    return v

In [ ]:
from src.models.gcn import gcn_resnet101, gcn_swin_t, gcn_swin_b
from src.models.addgcn import addgcn_resnet101
from src.models.chexnet import build_chexnet
from src.evaluate import evaluate, print_metrics
from src.baselines.train_addgcn import evaluate_addgcn

def build_gcn_model(cfg):
    backbone = cfg['model'].get('backbone', 'resnet101').lower()
    common_kwargs = {
        'num_classes': NUM_CLASSES,
        't': cfg['model']['t'],
        'pretrained': cfg['model'].get('pretrained', True),
        'adj_file': cfg['data']['adj'],
        'in_channel': cfg['model']['gcn_in'],
        'inp_file': cfg['data']['word_vec'],
    }
    if backbone == 'resnet101':
        return gcn_resnet101(**common_kwargs)
    if backbone == 'swin_t':
        return gcn_swin_t(**common_kwargs)
    if backbone == 'swin_b':
        return gcn_swin_b(**common_kwargs)
    raise ValueError(f'Unsupported backbone: {backbone}')

def build_dataset_and_loader(cfg, split='test', batch_size=None, workers=None):
    if split not in {'val', 'val_uncertain', 'test'}:
        raise ValueError('split must be val, val_uncertain, or test')

    if split == 'val':
        csv_key = 'val_csv'
    elif split == 'val_uncertain':
        csv_key = 'val_uncertain_csv'
    else:
        csv_key = 'test_csv'

    if csv_key not in cfg['data'] or not cfg['data'][csv_key]:
        raise ValueError(f'Missing {csv_key}.')

    dataset = CheXpert(
        root=cfg['data']['root'],
        csv_file=cfg['data'][csv_key],
        inp_name=cfg['data']['word_vec'],
        uncertain='keep',
        split='val',
    )

    bs = batch_size or cfg.get('train', {}).get('batch_size', 16)
    nw = workers if workers is not None else 0

    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=bs,
        shuffle=False,
        num_workers=nw,
        pin_memory=torch.cuda.is_available(),
    )
    return dataset, loader

In [ ]:
def evaluate_one_gcn_model(name, cfg_path, checkpoint_path, eval_split='test', data_root=None, device='cuda'):
    cfg = resolve_cfg(load_cfg(cfg_path), PROJECT_ROOT, data_root, test_csv_override=TEST_CSV)
    model = build_gcn_model(cfg).to(device)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = extract_state_dict(checkpoint)
    load_state_dict_flexible(model, state_dict)

    _, loader = build_dataset_and_loader(cfg, split=eval_split)
    results = evaluate(model, loader, device=device)

    print(f'\n===== {name} | split={eval_split} =====')
    print(f'checkpoint: {checkpoint_path}')
    print_metrics(results)

    return results

def evaluate_one_addgcn(name, cfg_path, checkpoint_path, eval_split='test', data_root=None, device='cuda'):
    cfg = resolve_cfg(load_cfg(cfg_path), PROJECT_ROOT, data_root, test_csv_override=TEST_CSV)
    model = addgcn_resnet101(num_classes=NUM_CLASSES, pretrained=False).to(device)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = extract_state_dict(checkpoint)
    load_state_dict_flexible(model, state_dict)

    _, loader = build_dataset_and_loader(cfg, split=eval_split)
    criterion = torch.nn.MultiLabelSoftMarginLoss().to(device)
    results = evaluate_addgcn(model, loader, criterion, device)

    print(f'\n===== {name} | split={eval_split} =====')
    print(f'checkpoint: {checkpoint_path}')
    print_metrics(results)

    return results

def evaluate_one_chexnet(name, cfg_path, checkpoint_path, eval_split='test', data_root=None, device='cuda'):
    cfg = resolve_cfg(load_cfg(cfg_path), PROJECT_ROOT, data_root, test_csv_override=TEST_CSV)
    
    model = build_chexnet(
        num_classes=NUM_CLASSES,
        pretrained=False,
    ).to(device)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = extract_state_dict(checkpoint)
    load_state_dict_flexible(model, state_dict)

    _, loader = build_dataset_and_loader(cfg, split=eval_split)
    results = evaluate(model, loader, device=device)

    print(f'\n===== {name} | split={eval_split} =====')
    print(f'checkpoint: {checkpoint_path}')
    print_metrics(results)

    return results

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EVAL_SPLIT = 'test'  # test-only, co the doi sang 'val' hoac 'val_uncertain' neu can

MODEL_SPECS = [
    {'name': 'C1', 'type': 'gcn',        'config': 'src/configs/c1.yaml'},
    {'name': 'C2', 'type': 'gcn',        'config': 'src/configs/c2.yaml'},
    {'name': 'C3', 'type': 'gcn',        'config': 'src/configs/c3.yaml'},
    {'name': 'C4', 'type': 'gcn',        'config': 'src/configs/c4.yaml'},
    {'name': 'C5', 'type': 'gcn',        'config': 'src/configs/c5.yaml'},
    {'name': 'AddGCN', 'type': 'addgcn', 'config': 'src/configs/addgcn_baseline.yaml'},
    {'name': 'CheXNet', 'type': 'chexnet', 'config': 'src/configs/chexnet_baseline.yaml'},
]

CHECKPOINTS = {
    'C1': 'https://drive.google.com/file/d/16V88n8bEHTlc497MicNuz4IKm-3ExE45/view?usp=sharing',
    'C2': 'https://drive.google.com/file/d/1-RnOnJIHbj3LKWnyXi4nfFSpx-3RyuPA/view?usp=drive_link',
    'C3': 'https://drive.google.com/file/d/1S8mTRB8f-34pAA5isKdZgkI4UzabzWi2/view?usp=drive_link',
    'C4': 'https://drive.google.com/file/d/1wjoL_wGV1_abtAtX-58YZDHlCDoIGPK1/view?usp=drive_link',
    'C5': 'https://drive.google.com/file/d/1uxH61z-zJVYU4gSt4S2dwvuY2kqup9Do/view?usp=drive_link',
    'AddGCN': 'https://drive.google.com/file/d/1J5XyIjQMlrF_5m4u1v1n2kyw_ozPMKuv/view?usp=drive_link',
    'CheXNet': 'https://drive.google.com/file/d/17-gvrYgHTjxwWTqyMQL87LHVOmibOMd2/view?usp=drive_link',
}

SEARCH_DIRS = [
    str(PROJECT_ROOT / 'checkpoints'),
    str(PROJECT_ROOT / 'ckpts'),
    '/kaggle/working/checkpoints',
]

print('DEVICE =', DEVICE)
print('EVAL_SPLIT =', EVAL_SPLIT)

In [ ]:
import time

rows = []
details = {}

splits = [EVAL_SPLIT]

for spec in MODEL_SPECS:
    name = spec['name']
    t0 = time.time()
    print(f'\n[INFO] Start model: {name}')
    cfg_path = str(PROJECT_ROOT / spec['config'])

    ckpt = resolve_checkpoint_entry(CHECKPOINTS.get(name), model_name=name, project_root=PROJECT_ROOT)

    if ckpt is not None and os.path.isdir(ckpt):
        ckpt = find_checkpoint([ckpt])

    if ckpt is None:
        candidates = [
            str(PROJECT_ROOT / 'checkpoints' / name.lower()),
            str(PROJECT_ROOT / 'checkpoints' / name),
            str(PROJECT_ROOT / 'checkpoints' / spec['name'].replace('C', 'c').replace('AddGCN', 'addgcn_baseline').replace('CheXNet', 'chexnet')),
            *SEARCH_DIRS,
        ]
        ckpt = find_checkpoint(candidates)

    if ckpt is None:
        print(f'[SKIP] {name}: khong tim thay checkpoint.')
        continue

    print(f'[INFO] Using checkpoint: {ckpt}')

    for split in splits:
        try:
            if spec['type'] == 'gcn':
                result = evaluate_one_gcn_model(
                    name=name,
                    cfg_path=cfg_path,
                    checkpoint_path=ckpt,
                    eval_split=split,
                    data_root=DATA_ROOT,
                    device=DEVICE,
                )
            elif spec['type'] == 'addgcn':
                result = evaluate_one_addgcn(
                    name=name,
                    cfg_path=cfg_path,
                    checkpoint_path=ckpt,
                    eval_split=split,
                    data_root=DATA_ROOT,
                    device=DEVICE,
                )
            elif spec['type'] == 'chexnet':
                result = evaluate_one_chexnet(
                    name=name,
                    cfg_path=cfg_path,
                    checkpoint_path=ckpt,
                    eval_split=split,
                    data_root=DATA_ROOT,
                    device=DEVICE,
                )
            else:
                raise ValueError(f"Unsupported model type: {spec['type']}")

            details[(name, split)] = result
            rows.append({
                'model': name,
                'split': split,
                'mAP': result.get('map'),
                'mean_auc': result.get('mean_auc'),
                'unc_auc': result.get('unc_auc'),
                'checkpoint': ckpt,
            })
        except Exception as ex:
            print(f'[ERROR] {name} ({split}): {ex}')

    print(f'[INFO] Done {name} in {(time.time() - t0):.1f}s')

summary_df = pd.DataFrame(rows)
if not summary_df.empty:
    summary_df = summary_df.sort_values(['split', 'mAP'], ascending=[True, False]).reset_index(drop=True)

summary_df

In [ ]:
out_dir = PROJECT_ROOT / 'results'
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / 'all_models_eval_summary.csv'

if 'summary_df' in globals() and not summary_df.empty:
    summary_df.to_csv(out_csv, index=False)
    print('Saved:', out_csv)
else:
    print('No evaluation results to save.')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if 'summary_df' not in globals() or summary_df.empty:
    print('Do not have summary_df')
else:
    try:
        plt.style.use('seaborn-v0_8-whitegrid')
    except Exception:
        plt.style.use('ggplot')

    plt.rcParams.update({
        'figure.facecolor': '#f7f8fb',
        'axes.facecolor': '#ffffff',
        'axes.edgecolor': '#d5d9e2',
        'axes.titleweight': 'bold',
        'axes.titlesize': 13,
        'axes.labelsize': 11,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'grid.alpha': 0.22,
        'grid.linestyle': '--',
        'font.family': 'DejaVu Sans',
    })

    metrics = ['mAP', 'mean_auc', 'unc_auc']
    metric_labels = {
        'mAP': 'mAP',
        'mean_auc': 'Mean AUC',
        'unc_auc': 'Uncertain AUC',
    }

    plot_df = summary_df.copy()
    for c in metrics:
        plot_df[c] = pd.to_numeric(plot_df[c], errors='coerce')

    # Dung model_split khi co nhieu split; neu test-only thi hien ten model cho gon
    if plot_df['split'].nunique() == 1:
        plot_df['display_name'] = plot_df['model']
    else:
        plot_df['display_name'] = plot_df['model'] + ' | ' + plot_df['split']

    # Loai bo dong metric rong
    plot_df = plot_df.dropna(subset=metrics, how='all')
    if plot_df.empty:
        print('Do not have any valid metric data to plot. Please check the evaluation results and checkpoint paths.')
    else:
        # Ranking score 
        rank_df = plot_df[['display_name', *metrics]].copy()
        for m in metrics:
            rank_df[f'rank_{m}'] = rank_df[m].rank(ascending=False, method='average')
        rank_df['rank_score'] = rank_df[[f'rank_{m}' for m in metrics]].mean(axis=1)
        rank_df = rank_df.sort_values('rank_score', ascending=True).reset_index(drop=True)

        # Figure 1: Leaderboard by metric 
        fig, axes = plt.subplots(1, 3, figsize=(22, 7))
        bar_colors = ['#2a9d8f', '#457b9d', '#e76f51']

        for i, m in enumerate(metrics):
            cur = plot_df[['display_name', m]].dropna().sort_values(m, ascending=True)
            axes[i].barh(cur['display_name'], cur[m], color=bar_colors[i], alpha=0.9)
            axes[i].set_title(f"{metric_labels[m]} Leaderboard")
            axes[i].set_xlabel(metric_labels[m])
            axes[i].set_xlim(0.0, 1.0)
            for y, v in enumerate(cur[m].tolist()):
                axes[i].text(min(v + 0.01, 0.98), y, f'{v:.3f}', va='center', fontsize=8)

        fig.suptitle('Model Leaderboard on Test Set', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # Figure 2: Heatmap (model x metric) 
        heat_df = rank_df.set_index('display_name')[metrics]
        fig, ax = plt.subplots(figsize=(9, max(4.5, 0.55 * len(heat_df))))
        im = ax.imshow(heat_df.values, aspect='auto', cmap='YlGnBu', vmin=0.0, vmax=1.0)

        ax.set_title('Metric Heatmap (Higher is Better)')
        ax.set_xticks(range(len(metrics)))
        ax.set_xticklabels([metric_labels[m] for m in metrics])
        ax.set_yticks(range(len(heat_df.index)))
        ax.set_yticklabels(heat_df.index)

        for r in range(heat_df.shape[0]):
            for c in range(heat_df.shape[1]):
                val = heat_df.values[r, c]
                color = 'black' if val < 0.70 else 'white'
                if pd.notna(val):
                    ax.text(c, r, f'{val:.3f}', ha='center', va='center', color=color, fontsize=9)

        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Score')
        plt.tight_layout()
        plt.show()

        # Figure 3: Radar chart top models
        top_n = min(5, len(rank_df))
        top_models = rank_df.head(top_n).copy()

        angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
        angles += angles[:1]

        fig = plt.figure(figsize=(8, 8))
        ax = plt.subplot(111, polar=True)
        radar_colors = ['#1d3557', '#2a9d8f', '#e9c46a', '#e76f51', '#6d597a']

        for idx, row in top_models.iterrows():
            vals = [row[m] if pd.notna(row[m]) else 0.0 for m in metrics]
            vals += vals[:1]
            ax.plot(angles, vals, linewidth=2.2, label=row['display_name'], color=radar_colors[idx % len(radar_colors)])
            ax.fill(angles, vals, alpha=0.08, color=radar_colors[idx % len(radar_colors)])

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels([metric_labels[m] for m in metrics])
        ax.set_ylim(0.0, 1.0)
        ax.set_title('Top Models Radar View', y=1.08, fontsize=14, fontweight='bold')
        ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), frameon=True)
        plt.tight_layout()
        plt.show()

        # Figure 4: Per-class AUC (lollipop) cho best model
        best_name = rank_df.iloc[0]['display_name']
        if ' | ' in best_name:
            best_model, best_split = best_name.split(' | ', 1)
        else:
            best_model = best_name
            best_split = plot_df['split'].iloc[0]

        best_key = (best_model, best_split)
        per_class_auc = details.get(best_key, {}).get('per_class_auc', {})

        if per_class_auc:
            auc_df = pd.DataFrame({
                'class': list(per_class_auc.keys()),
                'auc': list(per_class_auc.values()),
            }).sort_values('auc', ascending=True).reset_index(drop=True)

            y = np.arange(len(auc_df))
            fig, ax = plt.subplots(figsize=(11, max(5, 0.35 * len(auc_df))))
            ax.hlines(y=y, xmin=0, xmax=auc_df['auc'], color='#8ecae6', linewidth=2.5)
            ax.plot(auc_df['auc'], y, 'o', color='#023047', markersize=6)

            ax.set_yticks(y)
            ax.set_yticklabels(auc_df['class'])
            ax.set_xlim(0.0, 1.0)
            ax.set_xlabel('AUC')
            ax.set_title(f'Per-class AUC of Best Model: {best_model} ({best_split})')

            for yi, val in zip(y, auc_df['auc']):
                ax.text(min(val + 0.01, 0.98), yi, f'{val:.3f}', va='center', fontsize=8)

            plt.tight_layout()
            plt.show()
        else:
            print(f'[INFO] Khong co per_class_auc cho {best_key}')

        out_plot_dir = PROJECT_ROOT / 'results' / 'plots'
        out_plot_dir.mkdir(parents=True, exist_ok=True)
        print('Plot directory:', out_plot_dir)